In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import h5py
import tensorflow as tf
import numpy as np

from deep_lss.models.delta_model import DeltaLossModel
from deep_lss.nets.resnet import ResNetLayers
from msfm.utils import files

# np.set_printoptions(precision=5)


23-11-20 03:08:53   imports.py INF   Setting up healpy to run on 256 CPUs 


The goal of this notebook is to fix the following error:

```
Cannot use GPU when output.shape[1] * nnz(a) > 2^31 [Op:SparseTensorDenseMatMul]

Call arguments received by layer "chebyshev" (type Chebyshev):
  • input_tensor=tf.Tensor(shape=(208, 7264, 128), dtype=float32)
  • training=False
```

In [3]:
n_side = 512
data_vec_pix, _, _, _ = files.load_pixel_file()
n_neighbors = 20

# data
n_params = 6
n_batch = 4
n_batch *= (2 * n_params + 1)
n_pix = len(data_vec_pix)
n_z = 8
batch = tf.random.uniform((n_batch, n_pix, n_z), seed=10)
print(batch.shape)

23-11-10 01:46:39     files.py INF   Loaded the pixel file /global/u2/a/athomsen/multiprobe-simulation-forward-model/data/DESY3_pixels_512.h5 
(52, 464896, 8)


In [4]:
network = ResNetLayers(
    n_base_channels=32,
    # depth
    n_downsampling=3,
    n_cheby=2,
    n_residuals=5,
    # misc
    poly_degree=5,   
).get_layers()

In [5]:
# build the model
model = DeltaLossModel(
    network=network,
    n_side=n_side,
    indices=data_vec_pix,
    n_neighbors=n_neighbors,
    input_shape=(None, n_pix, n_z),
    max_batch_size=n_batch,
    checkpoint_dir=None,
    summary_dir=None,
    restore_checkpoint=False,
)

23-11-10 01:46:41 delta_model. INF   Initializing DeltaLossModel with a HealpyGCNN model 
Detected a reduction factor of 32.0, the input with nside 512 will be transformed to 16 during a forward pass. Checking for consistency with indices...
indices seem consistent...
Model: "healpy_gcnn_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 healpy_pseudo_conv (HealpyP  (None, 116224, 32)       1056      
 seudoConv)                                                      
                                                                 
 healpy_pseudo_conv_1 (Healp  (None, 29056, 64)        8256      
 yPseudoConv)                                                    
                                                                 
 healpy_pseudo_conv_2 (Healp  (None, 7264, 128)        32896     
 yPseudoConv)                                                    
                                                

In [6]:
# trace the operation
preds = model(batch)

In [7]:
%%timeit
# Get the last and second-to-last layers
last_layer = model.network.layers[-1]
second_to_last_layer = model.network.layers[-2]

# Create a new model with multiple outputs
new_model = tf.keras.Model(inputs=model.network.input, outputs=[last_layer.output, second_to_last_layer.output])

# new_preds = new_model(batch)

1.52 ms ± 11.3 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
# n_print = 10
# print(preds[:n_print].numpy())
# print(new_preds[0][:n_print].numpy())

# check output file

In [21]:
FILE = f"/pscratch/sd/a/athomsen/run_files/v5/debug/2023-11-10_00-41-40_resnet_vanilla/preds_500.h5"

with h5py.File(FILE, "r") as f:
    print(f.keys())
    print(f["fiducial"].keys())
    print(f["fiducial/vali"].keys())

    preds = f["fiducial/vali/pred"][:]


<KeysViewHDF5 ['fiducial']>
<KeysViewHDF5 ['vali']>
<KeysViewHDF5 ['i_example', 'i_noise', 'pred']>


In [25]:
FILE = f"/pscratch/sd/a/athomsen/run_files/v5/debug/2023-11-10_00-41-40_resnet_vanilla/preds_500_second_to_last.h5"

with h5py.File(FILE, "r") as f:
    print(f.keys())
    print(f["fiducial"].keys())
    print(f["fiducial/vali"].keys())

    preds_2 = f["fiducial/vali/pred"][:]
    second_to_last_layer = f["fiducial/vali/second_to_last_layer"][:]


<KeysViewHDF5 ['fiducial']>
<KeysViewHDF5 ['vali']>
<KeysViewHDF5 ['i_example', 'i_noise', 'pred', 'second_to_last_layer']>


In [23]:
preds[:10]

array([[-0.34271732,  0.01737615, -0.19433478,  0.08523756,  0.5506172 ,
         0.03985539,  0.218396  ],
       [-0.35123363,  0.04488083, -0.20962408,  0.08381038,  0.56779146,
         0.02961793,  0.24732189],
       [-0.348632  ,  0.01854857, -0.20696524,  0.08255415,  0.5661135 ,
         0.03223243,  0.23557787],
       [-0.33058944,  0.04252252, -0.19209364,  0.07311612,  0.5886221 ,
         0.0067741 ,  0.22867183],
       [-0.33182827,  0.03874084, -0.19505194,  0.07270365,  0.5850444 ,
         0.01254931,  0.23520927],
       [-0.3428661 ,  0.02290424, -0.20258406,  0.08540802,  0.5854392 ,
         0.03155198,  0.2327569 ],
       [-0.3435661 ,  0.01217267, -0.20080832,  0.08811908,  0.5694256 ,
         0.03888407,  0.23567371],
       [-0.3243753 ,  0.01902923, -0.20106009,  0.06991439,  0.59229374,
         0.00682702,  0.22461824],
       [-0.34610668,  0.04378864, -0.18814543,  0.06686168,  0.6011658 ,
         0.02084794,  0.2327569 ],
       [-0.3527061 ,  0.0326

In [24]:
preds_2[:10]

array([[-0.34271732,  0.01737615, -0.19433478,  0.08523756,  0.5506172 ,
         0.03985539,  0.218396  ],
       [-0.35173622,  0.04502233, -0.20904043,  0.08225112,  0.5676751 ,
         0.02843728,  0.24661855],
       [-0.348632  ,  0.01854857, -0.20696524,  0.08255415,  0.5661135 ,
         0.03223243,  0.23557787],
       [-0.330516  ,  0.04187306, -0.19262198,  0.07385903,  0.587821  ,
         0.0074946 ,  0.22852354],
       [-0.33694378,  0.04232666, -0.18932226,  0.07356101,  0.58067274,
         0.01188841,  0.23388939],
       [-0.3428661 ,  0.02290424, -0.20258406,  0.08540802,  0.5854392 ,
         0.03155198,  0.2327569 ],
       [-0.3486816 ,  0.01575848, -0.19507864,  0.08897644,  0.56505394,
         0.03822317,  0.23435383],
       [-0.32412067,  0.01969012, -0.20132521,  0.07117992,  0.5919237 ,
         0.00855174,  0.22558145],
       [-0.34471717,  0.0437214 , -0.18772009,  0.06630187,  0.59964466,
         0.01939931,  0.23116903],
       [-0.3527061 ,  0.0326

In [7]:
FILE = f"/pscratch/sd/a/athomsen/run_files/v5/lensing_only/2023-11-20_03-03-36_resnet_vanilla/preds_100.h5"

with h5py.File(FILE, "r") as f:
    print(f.keys())
    print(f["fiducial"].keys())
    print(f["fiducial/vali"].keys())

    i_example = f["fiducial/vali/i_example"][:]
    preds = f["fiducial/vali/pred"][:]


<KeysViewHDF5 ['fiducial']>
<KeysViewHDF5 ['vali']>
<KeysViewHDF5 ['i_example', 'i_noise', 'pred', 'second_to_last_layer']>


In [6]:
preds.shape

(40, 5)

In [8]:
i_example

array([ 11,  27,  59,  72, 112, 122, 137, 141, 143, 161, 289, 294, 319,
       325, 337, 350, 354, 355, 366, 389, 402, 425, 427, 444, 455, 489,
       505, 515, 543, 607, 645, 651, 655, 661, 665, 674, 727, 733, 752,
       783])